# 1. Imports

In [1]:
import requests
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from dash import Dash, dcc, html, Input, Output
from sklearn.linear_model import LogisticRegression

# 2. Chargement des données

In [2]:

# Veuillez remplacer le contenu de la variables "PATH" pointant vers les données dans votre cas.
PATH = "C:/Users/adama/3D Objects/obsidiant/iSheero/Hackathon/gdelt-benin/data/processed/GDELT_DATA_BENIN_2025-1.csv"

df = pd.read_csv(PATH)

df.head()

,SQLDATE,MentionTimeDate,GlobalEventID,Actor1Name,Actor2Name,EventCode,QuadClass,GoldsteinScale,AvgTone,ActionGeo_Fullname,MentionDocTone,Confidence,V2Organizations,V2Tone,MentionSourceName,SOURCEURL
0,20250101,20250101003000,1218370422,GOVERNOR,BENIN,16,1,-2.0,-12.206573,Benin,-12.206573,20,"House Of Assembly,528;House Of Assembly,1542;P...","-11.6822429906542,1.86915887850467,13.55140186...",punchng.com,https://punchng.com/pdp-seeks-igs-intervention...
1,20250101,20250101003000,1218370056,BENIN,NIGER,140,3,-6.5,-5.581395,Benin,-5.581395,100,Inconnu,"-5.14018691588785,2.33644859813084,7.476635514...",guardian.ng,https://guardian.ng/news/benin-protests-remark...
2,20250101,20250101003000,1218370660,NIGERIEN,BENIN,43,1,2.8,-5.581395,Benin,-5.581395,20,Inconnu,"-5.14018691588785,2.33644859813084,7.476635514...",guardian.ng,https://guardian.ng/news/benin-protests-remark...
3,20250101,20250101003000,1218370058,BENINESE,NIGERIEN,42,1,1.9,-5.581395,Benin,-5.581395,80,Inconnu,"-5.14018691588785,2.33644859813084,7.476635514...",guardian.ng,https://guardian.ng/news/benin-protests-remark...
4,20250101,20250101003000,1218370051,BENIN,Inconnu,20,1,3.0,-12.206573,Benin,-12.206573,100,"House Of Assembly,528;House Of Assembly,1542;P...","-11.6822429906542,1.86915887850467,13.55140186...",punchng.com,https://punchng.com/pdp-seeks-igs-intervention...


# 3. Prétraitement

### 3.1. Suppression des valeur maquante

In [3]:
df["date"] = pd.to_datetime(df["SQLDATE"], format="%Y%m%d", errors="coerce") # Formatage de des dates.
df["tone"] = pd.to_numeric(df["AvgTone"], errors="coerce") # Numérisation des valeurs moyennes des tons.
df["impact"] = pd.to_numeric(df["GoldsteinScale"], errors="coerce") # Numérisation de l'échelle de Goldstein.

df = df.dropna(subset=["date"])

### 3.2. liste des codes valides

In [4]:
VALID_CODES = {f"{i:02d}" for i in range(1, 21)}

### 3.3. Nettoyage

In [5]:
df["EventRootCode"] = (
    df["EventCode"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(3)
    .str[:2]
)

df = df[df["EventRootCode"].isin(VALID_CODES)]

df["EventRootCode"] = df["EventRootCode"].where(
    df["EventRootCode"].isin(VALID_CODES),
    "00"
)

### 3.4. Mapping

In [6]:
EVENT_MAP_FR = {
    "01": "Déclaration publique",
    "02": "Appel",
    "03": "Intention de coopérer",
    "04": "Consultation",
    "05": "Coopération",
    "06": "Aide",
    "07": "Soutien",
    "08": "Enquête",
    "09": "Demande",
    "10": "Désaccord",
    "11": "Critique",
    "12": "Rejet",
    "13": "Menace",
    "14": "Protestation",
    "15": "Démonstration",
    "16": "Violence",
    "17": "Coercition",
    "18": "Attaque",
    "19": "Conflit",
    "20": "Autre",
}

df["EventCategory"] = (
    df["EventRootCode"]
    .map(EVENT_MAP_FR)
    .fillna("Inconnu")
)

# 4. Analyse exploratoire

## 4.1 Timeline des événements

In [7]:
timeline = (
    df.groupby(df["date"].dt.date)
    .size()
    .reset_index(name="count")
    .sort_values("date")
)

fig = px.line(
    timeline,
    x="date",
    y="count",
    title="Évolution des événements au Bénin"
)

fig.show()

### Ce que ça signifie :

- évolution du nombre d’événements dans le temps
- donc : activité géopolitique au Bénin

## 4.2 Top acteurs

In [8]:
actors = (
    df["Actor1Name"]
    .value_counts()
    .head(10)
    .reset_index()
)

actors.columns = ["actor", "count"]

fig = px.bar(
    actors,
    x="actor",
    y="count",
    title="Top 10 des acteurs",
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

### Signification :

- Qui agit le plus dans les événements
- ex : gouvernement, ONG, etc.


## 4.3 Types d’événements

In [9]:


df["EventCategory"] = (
    df["EventRootCode"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(2)
    .str[:2]
    .map(EVENT_MAP_FR)
    .fillna("Inconnu")
)

event_counts = (
    df["EventCategory"]
    .value_counts()
    .reset_index()
)

event_counts.columns = ["Catégorie", "Nombre"]

fig = px.bar(
    event_counts,
    x="Catégorie",
    y="Nombre",
    title="Distribution des types d'événements"
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

### Quels types d’événements dominent ?

Exemples :

* protest
* conflict
* cooperation

## 4.4 Quel acteur fait quel type d’événement ?

In [10]:
cross = (
    df.groupby(["Actor1Name", "EventCategory"])
    .size()
    .reset_index(name="count")
)

top_actors = df["Actor1Name"].value_counts().head(5).index

cross = cross[cross["Actor1Name"].isin(top_actors)]

fig = px.bar(
    cross,
    x="Actor1Name",
    y="count",
    color="EventCategory",
    title="Types d'événements par acteur"
)

fig.show()

### Ce que ça apporte

Tu peux dire :

- "La police est surtout liée aux protestations"
- "Le gouvernement est lié aux déclarations publiques"
- "Certains acteurs sont liés à la violence"

## 4.5 Distribution du ton

In [11]:
fig = px.histogram(
    df,
    x="tone",
    nbins=50,
    title="Distribution du ton (sentiment)",
)

fig.show()

### Interprétation

| valeur | signification                  |
| ------ | ------------------------------ |
| > 0    | positif (stable / coopération) |
| < 0    | négatif (conflits)             |
| 0      | neutre                         |

Ce n’est pas un graphique direct, c’est une statistique globale

# 5. Machine Learning

## 5.1 Création cible

In [12]:
df["target"] = df["tone"] > 0

## 5.2 Entraînement modèle

### 5.2.1. Est-ce que GoldsteinScale permet de prédire un ton positif ou négatif ?

Pour comprendre ce que sait l'échelle de Goldstein (GoldsteinScale), rendez-vous au lien : [clic-moi](http://data.gdeltproject.org/documentation/CAMEO.Manual.1.1b3.pdf)

In [13]:
X = df[["GoldsteinScale", "EventCode"]]
y = df["AvgTone"] > 0
model = LogisticRegression()
model.fit(X, y)

LogisticRegression()

## 5.3 Prédiction

In [14]:
df["prediction"] = model.predict(X)

df[["impact", "tone", "target", "prediction"]].head()

,impact,tone,target,prediction
0,-2.0,-12.206573,False,False
1,-6.5,-5.581395,False,False
2,2.8,-5.581395,False,False
3,1.9,-5.581395,False,False
4,3.0,-12.206573,False,False


## 5.4 Visualisation ML

In [15]:
fig = px.scatter(
    df,
    x="impact",
    y="tone",
    color="target",
    title="Relation Impact vs Sentiment",
)

fig.show()

In [16]:
df["label"] = df["AvgTone"].apply(lambda x: "positive" if x > 0 else "negative")

fig = px.scatter(
    df,
    x="GoldsteinScale",
    y="AvgTone",
    color="label",
    color_discrete_map={
        "positive": "green",
        "negative": "red"
    },
    title="Relation Goldstein vs Tone (GDELT Bénin)"
)

fig.show()


### 5.5. Insight important
- Cluster rouge = zones de tension
- Cluster vert = stabilité / coopération
- L’axe X = impact géopolitique (GoldsteinScale)
- L’axe Y = tonalité médiatique (AvgTone)

# 6. Insights

### Insights clés

- Les événements sont concentrés sur certaines périodes → pics d'activité
- Certains acteurs dominent fortement les interactions
- Le sentiment global est (positif / négatif selon tes données)
- L’impact (GoldsteinScale) influence partiellement le ton

### ML

- Modèle simple mais fonctionnel
- Permet de prédire le ton à partir de l’impact
- Peut être amélioré avec plus de variables

# 7. Conclusion

### Conclusion

Ce projet démontre :

- Un pipeline complet de données (ingestion → analyse)
- Une API exploitable
- Des visualisations claires
- Une première approche Machine Learning

Ce système peut être étendu pour :
- monitoring en temps réel
- détection de crises
- analyse géopolitique avancée

In [17]:

# df = pd.read_csv(PATH, sep=",", encoding="latin1")

# df["SQLDATE"] = pd.to_datetime(df["SQLDATE"], format="%Y%m%d")

# app = Dash(__name__)

# app.layout = html.Div([
#     html.H1("Dashboard GDELT - Crises et Relations"),

#     dcc.DatePickerRange(
#         id="date-picker",
#         start_date=df["SQLDATE"].min(),
#         end_date=df["SQLDATE"].max()
#     ),

#     dcc.Dropdown(
#         options=[{"label": str(i), "value": i} for i in sorted(df["QuadClass"].dropna().unique())],
#         multi=True,
#         id="quad-filter",
#         placeholder="Filtrer par type d'événement"
#     ),

#     html.Label("Acteur 1"),
#     dcc.Dropdown(
#         options=[{"label": i, "value": i} for i in df["Actor1Name"].dropna().unique()],
#         multi=True,
#         id="actor1-filter"
#     ),

#     html.Label("Recherche"),
#     dcc.Input(
#         id="search",
#         type="text",
#         placeholder="ex: gouvernement, police..."
#     ),

#     dcc.Graph(id="time-series"),
#     dcc.Graph(id="map"),
#     dcc.Graph(id="relations")
# ])

# @app.callback(
#     [Output("time-series", "figure"),
#      Output("map", "figure"),
#      Output("relations", "figure")],
#     [Input("date-picker", "start_date"),
#      Input("date-picker", "end_date"),
#      Input("quad-filter", "value"),
#      Input("actor1-filter", "value"),
#      Input("search", "value")]
# )
# def update_graphs(start_date, end_date, quad_values, actors, search):

#     dff = df.copy()

#     dff = dff[(dff["SQLDATE"] >= start_date) & (dff["SQLDATE"] <= end_date)]

#     if quad_values:
#         dff = dff[dff["QuadClass"].isin(quad_values)]

#     if actors:
#         dff = dff[dff["Actor1Name"].isin(actors)]

#     if search:
#         dff = dff[dff["Actor1Name"].str.contains(search, case=False, na=False)]

#     if dff.empty:
#         empty_fig = px.scatter(title="Aucune donnée disponible")
#         return empty_fig, empty_fig, empty_fig

#     df_daily = dff.groupby("SQLDATE").agg({
#         "GoldsteinScale": "mean",
#         "AvgTone": "mean"
#     }).reset_index()

#     fig_time = px.line(
#         df_daily,
#         x="SQLDATE",
#         y=["GoldsteinScale", "AvgTone"],
#         title="Évolution des crises"
#     )

#     try:
#         dff_map = dff.dropna(subset=["ActionGeo_Fullname"])

#         fig_map = px.scatter_geo(
#             dff_map,
#             locations="ActionGeo_Fullname",
#             locationmode="country names",
#             color="AvgTone",
#             size="GoldsteinScale",
#             hover_name="ActionGeo_Fullname",
#             title="Carte des événements"
#         )
#     except:
#         fig_map = px.scatter(title="Carte indisponible")

#     try:
#         rel = dff[["Actor1Name", "Actor2Name"]].dropna()

#         if rel.empty:
#             fig_rel = px.scatter(title="Pas de relations disponibles")
#         else:
#             rel = rel.value_counts().reset_index(name="count").head(15)
#             rel["relation"] = rel["Actor1Name"] + " → " + rel["Actor2Name"]

#             fig_rel = px.bar(
#                 rel,
#                 x="count",
#                 y="relation",
#                 orientation='h',
#                 title="Relations entre acteurs"
#             )
#     except:
#         fig_rel = px.scatter(title="Erreur relations")

#     return fig_time, fig_map, fig_rel

# if __name__ == "__main__":
#     app.run(
#         host="127.0.0.1",
#         port=8050,
#         debug=True
#     )
